# AIMENG：Google Colab 免费 GPU 训练入口

本 notebook 面向免费 Colab GPU。它先检查 GPU、磁盘与内存，再获取固定的 Ornith GGUF 并生成 manifest。**GGUF 本身作为冻结推理基线，不进行假装可行的常规反向传播。**

真正训练路由器需要来自实际后端的测量轨迹：每个样本至少包含可观测特征、候选通道组/候选池、质量分数、延迟和峰值内存。没有真实轨迹时，只运行环境检查，不会伪造训练数据或宣称训练成功。

In [ ]:
import os, sys, shutil, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('RAM (GiB):', round(__import__('psutil').virtual_memory().total / 2**30, 2))
print('Free disk (GiB):', round(shutil.disk_usage('/content').free / 2**30, 2))
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as e:
    print('PyTorch/GPU probe failed:', repr(e))
assert shutil.disk_usage('/content').free > 8 * 2**30, 'Colab 可用磁盘不足 8 GiB；不要开始下载。'


## GPU 显存与吞吐量策略
目标是 16 GiB GPU 上约 14 GiB 的总显存占用，但采用自适应策略，不硬编码分配 14 GiB 张量。优先增加有效 batch / 梯度累积、适度增加序列长度，并在训练真实路由器时使用 BF16（硬件支持时）或 FP16、混合精度和高效 DataLoader。若 GPU 小于 16 GiB，则自动缩小 batch/序列长度。模型下载和 GGUF 推理占用不等于训练路由器的训练显存。

In [ ]:
import torch, json
def gpu_memory_plan(target_gib=14.0, reserve_gib=2.0):
    if not torch.cuda.is_available():
        print('没有检测到 CUDA GPU；先停止，不会伪装成 GPU 训练。')
        return None
    p = torch.cuda.get_device_properties(0)
    total = p.total_memory / 2**30
    free, _ = torch.cuda.mem_get_info(0)
    free_gib = free / 2**30
    target = min(target_gib, max(0.0, total - reserve_gib))
    print(json.dumps({'gpu': p.name, 'total_vram_gib': round(total, 2), 'currently_free_gib': round(free_gib, 2), 'target_total_usage_gib': round(target, 2), 'reserve_gib': round(min(reserve_gib, total), 2), 'note': 'planning ceiling only; never allocate dummy tensors'}, ensure_ascii=False, indent=2))
    if total >= 15.5:
        print('Profile: 16-GiB class; tune real batch/sequence length using measured throughput.')
    else:
        print('Profile: smaller GPU; use conservative settings and do not force 14 GiB allocation.')
    return {'total_gib': total, 'free_gib': free_gib, 'target_gib': target}
GPU_PLAN = gpu_memory_plan()


### 性能验收原则
- 不使用占位张量把显存人为塞满；这只会挤掉真实训练需要的显存，降低吞吐并增加 OOM 风险。
- 训练正式路由器时先测 batch size 1/2/4/8 的 step time 与 tokens/s，选择最快且留有安全余量的配置；如果支持 BF16，优先测试 BF16。
- 以有效训练样本/秒、GPU 利用率和稳定性判断，而不是只看显存占用数字。
- Colab 会话可能被回收；按固定间隔保存路由器 checkpoint 与训练状态到 Google Drive，避免会话结束后从头来过。
- 当前 notebook 的主要工作仍是环境检查和 GGUF 元数据核验；没有真实后端轨迹时，不会启动无意义的路由训练。

## 1. 获取仓库代码
在 Colab 菜单选择 **运行时 → 更改运行时类型 → T4 GPU（如果账号当时提供）**。免费 GPU 的可用性、型号和时长由 Google 动态分配，并不保证每次都有。

In [ ]:
import os, subprocess
REPO = 'https://github.com/bslsjdk/aimeng.git'
BRANCH = 'feat/gguf-router-training-foundation'
if not os.path.isdir('/content/aimeng/.git'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/aimeng'], check=True)
else:
    subprocess.run(['git','-C','/content/aimeng','fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C','/content/aimeng','checkout',BRANCH], check=True)
%cd /content/aimeng
!python -m pip install -q -r requirements.txt psutil

## 2. 下载并核对 GGUF
文件约 5.8 GB。Colab 免费实例的磁盘空间会变化；若空间不足，停止而不是反复下载。模型不会上传到 Git。

In [ ]:
from pathlib import Path
free_gib = shutil.disk_usage('/content').free / 2**30
print(f'当前剩余磁盘：{free_gib:.2f} GiB')
if free_gib < 8:
    raise RuntimeError('磁盘空间不足 8 GiB，已停止，避免下载中断。')
subprocess.run([sys.executable, 'scripts/fetch_model.py'], check=True)
subprocess.run([sys.executable, 'scripts/inspect_gguf.py', '--output', 'runs/gguf_manifest.json'], check=True)
print(Path('runs/gguf_manifest.json').read_text()[:6000])

## 3. 训练前数据门槛（当前不会自动训练）
本仓库当前还没有已经验证的、能在该 9B 后端中真实跳过 FFN 计算的执行后端，也没有可证明有效的训练轨迹生成器。因此这个 notebook 当前能完成的是：检查 Colab、下载并核验 GGUF、记录模型 manifest。**它不会训练 9B 权重，也不会凭普通问答文本训练出可信的神经元路由器。**

下一阶段必须先实现并验证实际后端采集器：同一任务分别跑 small / medium / full 预算，记录质量、延迟、峰值资源和后端实际采用参数。只有真实轨迹通过 schema 检查后，才能训练独立的小型预算控制器。

In [ ]:
from pathlib import Path
trace = Path('data/telemetry.jsonl')
validator = Path('scripts/validate_telemetry.py')
if not trace.exists():
    print('STOP: 尚无 data/telemetry.jsonl。当前运行到这里是预期行为，不是训练失败。')
    print('下一步工程任务是实现并验证后端遥测采集器；不要创建随机数据来绕过这个门槛。')
else:
    subprocess.run([sys.executable, str(validator), str(trace), '--check-task-split-leakage'], check=True)
    print('数据格式检查通过不等于标签真实性已验证，也不等于控制器训练已完成。')


## 4. 本次运行的验收标准
- 环境检查成功，且确实检测到 CUDA GPU（否则没有开始 GPU 训练）。
- GGUF 下载成功，`runs/gguf_manifest.json` 已生成，SHA-256 与模型元数据已记录。
- 如果没有真实 `data/telemetry.jsonl`，本次就到此为止；这只是准备检查，不是训练。
- 有遥测数据时，先运行 validator 并人工核实测量来源，再开发/运行控制器训练脚本。
- 后续控制器必须在独立留出任务上与规则基线比较质量、延迟、吞吐、峰值内存和失败率。
- Android 端单独测整应用峰值 RAM，硬限制仍是 4096 MiB，不能用 Colab 显存结果替代。

不要把环境检查、模型下载或数据格式检查写成“路由器训练成功”。

## 打开后具体怎么操作
1. 打开本 notebook，登录 Google 账号。
2. 菜单选择 **运行时 → 更改运行时类型 → T4 GPU（若免费额度当前提供）**，保存。
3. 选择 **运行时 → 全部运行**。首次运行会安装依赖、检查资源并下载约 5.8 GB 的 GGUF；保持页面打开，直到各单元结束。
4. 若磁盘检查失败，停止，不要反复点运行；换一个有足够可用磁盘的 Colab 会话再试。
5. 检查 `runs/gguf_manifest.json` 是否生成，确认输出中 `status: ok`、文件名和 SHA-256。
6. 如果最后显示缺少 `data/telemetry.jsonl`，这是当前预期的停止点。不要伪造数据；真正训练要等后端遥测采集器和控制器训练脚本完成。
7. Colab 临时文件可能随会话回收而丢失；在真正开始训练前，必须再配置 Google Drive 持久化 checkpoint 与数据。

**重要：当前 notebook 不是一键训练器。** 它是环境与模型检查入口。训练按钮式流程还需完成后端数据采集器、训练脚本、验证与 checkpoint 保存。